In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import pyvinecopulib as pv
from scipy.stats import genpareto
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [5]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [6]:
class MarginalTailModel:
  def __init__(self, sim_params, ticker):
    self.sim_params = sim_params
    self.ticker = ticker
    self.params = None

  def fit_dist(self, r):
    full_idx = r.index
    r = r.dropna()
    valid_idx = r.index
    r = r.to_numpy()

    q_upper = self.sim_params.q_upper
    q_lower = 1 - self.sim_params.q_upper

    u_upper = np.percentile(r, q_upper * 100)
    u_lower = np.percentile(r, q_lower * 100)

    upper_tail = r[r > u_upper]
    lower_tail = r[r < u_lower]

    c_U, _, scale_U = genpareto.fit(upper_tail - u_upper, floc=0)
    c_L, _, scale_L = genpareto.fit(u_lower - lower_tail, floc=0)

    p_l = np.mean(r < u_lower)
    p_u = np.mean(r > u_upper)

    lower_mask = r < u_lower
    upper_mask = r > u_upper
    body_mask = ~lower_mask & ~upper_mask

    u_resid = np.zeros_like(r, dtype=float)

    if np.any(lower_mask):
      cdf = 1 - genpareto.cdf(u_lower - r[lower_mask], c_L, scale=scale_L)
      u_resid[lower_mask] = p_l * cdf

    if np.any(body_mask):
      body = r[body_mask]
      ranks = pd.Series(body).rank(method='average').to_numpy()
      cdf = p_l + (1 - p_l - p_u) * (ranks / (len(body) + 1))
      u_resid[body_mask] = cdf

    if np.any(upper_mask):
      cdf = (1 - p_u) + p_u * genpareto.cdf(r[upper_mask] - u_upper, c_U, scale=scale_U)
      u_resid[upper_mask] = cdf

    self.params = {
      "c_L": c_L,
      "scale_L": scale_L,
      "c_U": c_U,
      "scale_U": scale_U,
      "p_l": p_l,
      "p_u": p_u,
      "u_lower": u_lower,
      "u_upper": u_upper,
      "ecdf_data": r[(r >= u_lower) & (r <= u_upper)]
    }

    u_resid_full = pd.Series(np.nan, index=full_idx)
    u_resid_full.loc[valid_idx] = u_resid
    return u_resid_full

  def inverse_cdf(self, u_resid):
    z_resid_final = np.zeros_like(u_resid, dtype=float)

    for i in range(u_resid.shape[1]):
      u_resid_i = u_resid[:, i]
      real_col = np.zeros_like(u_resid_i)

      p_u = self.params["p_u"]
      p_l = self.params["p_l"]

      lower_mask = u_resid_i < p_l
      upper_mask = u_resid_i > (1 - p_u)
      body_mask = (~lower_mask) & (~upper_mask)

      if np.any(upper_mask):
        ppf_l = (
          self.params["u_upper"] +
          genpareto.ppf(
            (u_resid_i[upper_mask] - (1 - p_u))/p_u,
            self.params["c_U"],
            scale=self.params["scale_U"]
          )
        )
        real_col[upper_mask] = ppf_l

      if np.any(lower_mask):
        ppf_l = (
          self.params["u_lower"] -
          genpareto.ppf(
            1 - u_resid_i[lower_mask]/p_l,
            self.params["c_L"],
            scale=self.params["scale_L"]
          )
        )

        real_col[lower_mask] = ppf_l

      if np.any(body_mask):
        body = (u_resid_i[body_mask] - p_l) / (1 - p_l - p_u) * 100
        body_percentiles = np.clip(body, 0, 100)
        ppf_b = np.percentile(self.params["ecdf_data"], body_percentiles)
        real_col[body_mask] = ppf_b

      z_resid_final[:, i] = real_col

    return z_resid_final

In [144]:
class GARCHEVTCOPULA:
  def __init__(self, sim_params, debug:bool=False, **kwargs):
    self.sim_params = sim_params
    self.debug = debug
    self.copula = None
    self.model = None
    self.vol_init = {}
    self.r_init = {}
    self.best_models = {}
    self.best_fits = {}
    self.best_params = {}
    self.scale_factor = 100
    self.marginal_distributions = {}

  def _fit_model(self, r):
    best_bic = np.inf
    for lags in range(1, self.sim_params.lags+1):
      model = arch_model(
        r,
        mean="AR",
        lags=lags,
        vol="GARCH",
        p=1,
        o=1,
        q=1,
        dist='studentst'
      )

      fit = model.fit(disp='off', show_warning=False)

      if fit.bic < best_bic and not fit.convergence_flag:
        best_bic = fit.bic
        best_model, best_fit, best_lags = model, fit, lags

    return best_model, best_fit, best_lags


  def _fit_arma_garch(self, returns):
    print("------------- Fitting AR-GARCH -------------")
    log_returns = np.log1p(returns)
    filtered_resid = pd.DataFrame(index=returns.index, columns=returns.columns)

    for i, col in enumerate(log_returns.columns):
      r_scaled = log_returns[col] * self.scale_factor
      self.vol_init[col] = r_scaled.std()
      self.r_init[col] = r_scaled.mean()

      best_model, best_fit, best_lags = self._fit_model(r_scaled)

      self.best_models[col] = best_model
      self.best_fits[col] = best_fit
      self.best_params[col] = best_fit.params

      filtered_resid[col] = best_fit.std_resid

    return filtered_resid


  def _get_uniform_residuals(self, residuals):
    print("------------ Uniform Residuals -------------")
    u_resid = pd.DataFrame(index=residuals.index, columns=residuals.columns)
    for ticker in residuals.columns:
      model = MarginalTailModel(self.sim_params, ticker)
      u_resid_i = model.fit_dist(residuals[ticker])
      self.marginal_distributions[ticker] = model

      u_resid[ticker] = u_resid_i

    return u_resid

  def _inverse_semi_parametric_cdf(self, uniform_samples):
    print("------------ Inverse Semi Parametric CDF -------------")
    n_paths, n_steps, n_assets = uniform_samples.shape
    z_resid = np.zeros((n_paths, n_steps, n_assets), dtype=float)

    for asset_idx, ticker in enumerate(self.cols):
      model = self.marginal_distributions[ticker]
      u_slice = uniform_samples[:, :, asset_idx].T
      z_slice = model.inverse_cdf(u_slice)
      z_resid[:, :, asset_idx] = z_slice.T

    return z_resid


  def _fit_vine_copula(self, residuals):
    print("-------------- Fitting Copula --------------")
    np_resid = residuals.dropna().to_numpy()
    controls = pv.FitControlsVinecop()
    self.copula = pv.Vinecop.from_data(np_resid, controls=controls)


  def fit(self, returns):
    z_resid = self._fit_arma_garch(returns)
    self.cols = z_resid.columns

    u_resid = self._get_uniform_residuals(z_resid)

    self._fit_vine_copula(u_resid)

  def generate_sample(self):
    print("------------ Generating Samples ------------")
    n_paths = self.sim_params.n_paths
    n_steps = self.sim_params.n_steps
    n_assets = len(self.best_models)

    total_obs = n_paths * n_steps
    simulated_steps = self.copula.simulate(n=total_obs)
    simulated_uniforms = simulated_steps.reshape(n_paths, n_steps, n_assets)

    simulated_residuals = self._inverse_semi_parametric_cdf(simulated_uniforms)
    simulated_residuals = np.transpose(simulated_residuals, (0, 2, 1))

    simulated_returns = np.zeros_like(simulated_residuals)
    for i, ticker in enumerate(self.cols):
      simulated_residuals[:, i, :] = simulated_residuals[:, i, :]
      params = self.best_params[ticker]
      model = self.best_models[ticker]

      log_paths = self._convert_to_returns(
        simulated_residuals[:, i, :],
        self.r_init[ticker],
        model,
        params,
        ticker
      )

      simulated_returns[:, i, :] = np.exp(log_paths/self.scale_factor) - 1

    return simulated_returns


  def _convert_to_returns(
    self,
    simulated_residuals,
    init_states,
    model,
    params,
    ticker
  ):
    print(f"------ Converting {ticker} to Returns -----")
    lags = model.lags

    const = params["Const"]
    ar = params.get(f"{ticker}{lags[0]}", 0.0)
    ma = params.get(f"{ticker}{lags[1]}", 0.0)
    omega = params["omega"]
    alpha = params["alpha[1]"]
    gamma = params["gamma[1]"]
    beta = params["beta[1]"]
    nu = params["nu"]

    r_t = np.zeros_like(simulated_residuals)

    sigma2 = np.zeros_like(simulated_residuals)
    sigma2_prev = self.vol_init[ticker]**2
    r_prev = self.r_init[ticker]
    eps_prev = self.vol_init[ticker] * simulated_residuals[:, 0]

    for t in range(simulated_residuals.shape[-1]):
      I = np.where(eps_prev < 0, 1.0, 0.0)
      sigma2_prev
      sigma2[:, t] = (
          omega
          + (alpha + gamma * I) * eps_prev**2
          + beta * sigma2_prev
        )
      sigma2_prev = sigma2[:, t]

      eps_t = np.sqrt(sigma2[:, t]) * simulated_residuals[:, t]

      ar_term = np.sum(ar*r_prev) if isinstance(ar, np.ndarray) else ar*r_prev
      ma_term = np.sum(ma*eps_t) if isinstance(ma, np.ndarray) else ma*eps_t

      mu_t = const + ar_term + ma_term
      r_t[:, t] = mu_t + eps_t

      r_prev = r_t[:, t]
      eps_prev = eps_t

    return r_t



In [148]:
@dataclass
class SimParams:
  q_upper:float=0.95
  q_lower:float=0.95
  n_paths:int = 10000
  n_steps:int = 100
  lags:int = 5

@dataclass
class CVaRParams:
  alpha_threshold:float=0.95
  optimize = True


In [ ]:
class CVaREngine(GARCHEVTCOPULA, DataStore):
  def __init__(
      self,
      sim_params:SimParams,
      cvar_engine_params:CVaRParams,
      debug:bool=False,
      **kwargs):
    super().__init__(
      debug=debug,
      sim_params=sim_params,
      **kwargs
    )
    self.debug = debug
    self.cvar_params = cvar_engine_params

  def get_data(self, universe, start, end):
    data, benchmark = self._get_data(universe=universe, start=start, end=end)
    returns = data.pct_change().dropna()

    return returns, benchmark


  def fit_garch_evt_copula(self, returns):
    self.universe = returns.columns
    self.fit(returns)

  def generate_r_sample(self, _return=False):
    sim_returns = self.generate_sample()
    if _return:
      return self.sim_returns
    else:
      self.sim_returns = sim_returns

  def calculate_mcvar(self, sim_returns, w=None):
    if w is None and not self.cvar_params.optimize:
      w = np.ones(len(self.universe)) / len(self.universe)

    ptf = np.tensordot(sim_returns, w, axes=([1], [0]))
    R = np.prod(1 + ptf, axis=1) - 1

    L = -R
    alpha = self.cvar_params.alpha_threshold * 100
    var_p = np.percentile(L, alpha)
    tail_mask = L >= var_p
    cvar_p = np.mean(L[tail_mask])

    asset_R = np.prod(1 + sim_returns, axis=2) - 1
    asset_L = -asset_R

    tail_losses = asset_L[tail_mask]
    mcvar = np.mean(tail_losses, axis=0)

    return mcvar, cvar_p

In [74]:
sim_params = SimParams()

In [75]:
eng = CVaREngine(sim_params)

In [76]:
returns, benchmark = eng.get_data(
    universe=test_universe,
    start="2019-01-01",
    end="2027-01-01"
)

In [77]:
eng.fit(returns)

------------- Fitting AR-GARCH -------------
------------ Uniform Residuals -------------
-------------- Fitting Copula --------------


In [78]:
resid = eng.generate_sample()

------------ Generating Samples ------------
------------ Inverse Semi Parametric CDF -------------
------------ Converting BNDW to Returns ------------
------------ Converting DBMF to Returns ------------
------------ Converting USCI to Returns ------------
------------ Converting XLK to Returns ------------
------------ Converting XLP to Returns ------------
------------ Converting XLV to Returns ------------
